# Quantum-Assisted Edge–Cloud Collaborative Task Scheduling

End-to-end implementation of the workflow in the supplied methodology:

1. Workload acquisition and schema normalization (Alibaba/Google/Azure/Bitbrains files or the linked Kaggle dataset).
2. DAG reconstruction, deadline/QoS characterization, and heterogeneous resource profiling.
3. Multi-Relational Task–Resource Hypergraph (MR-TRH) construction and partitioning.
4. Hypergraph-guided QUBO formulation, optional QAOA execution with Qiskit Aer, and classical feasibility repair.
5. Adaptive Hypergraph-guided Large-Neighborhood Search (AHLNS).
6. Baseline comparisons, ablation, scalability, five-fold evaluation, statistical tests, feasibility analysis, SHAP/LIME explanations, plots, and a lightweight scheduling interface.

The notebook uses actual records from the selected workload files. It does not silently fabricate workload records. Where the Kaggle dataset schema differs, the normalization cell maps available columns and reports fields that must be supplied or inferred from documented columns.

## 1. Environment

Run this notebook in Kaggle or Colab. If Qiskit, SHAP, or LIME are unavailable, the notebook installs them. The QAOA section uses Qiskit Aer when installed; a classical QUBO solver remains available for reproducibility and fallback. Simulated annealing is explicitly treated as a classical solver, not quantum hardware.

In [ ]:
import os, sys, glob, json, time, math, random, warnings, subprocess, importlib.util
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy.stats import ttest_rel, wilcoxon
from scipy.optimize import milp, LinearConstraint, Bounds
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.inspection import permutation_importance

SEED = 42
random.seed(SEED); np.random.seed(SEED)
SAVE_DIR = "Scopus_VS76"
RESULT_DIR = os.path.join(SAVE_DIR, "results")
os.makedirs(RESULT_DIR, exist_ok=True)

def install_if_missing(package, import_name=None):
    name = import_name or package
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for pkg, mod in [("qiskit", "qiskit"), ("qiskit-aer", "qiskit_aer"),
                 ("qiskit-optimization", "qiskit_optimization"),
                 ("shap", "shap"), ("lime", "lime")]:
    try:
        install_if_missing(pkg, mod)
    except Exception as e:
        print(f"Optional dependency unavailable: {pkg}: {e}")

print("Results directory:", RESULT_DIR)

## 2. Workload acquisition

Set `DATA_ROOT` to the directory containing the Kaggle download or trace files. For Kaggle, add the dataset to the notebook through **Add Input** and point `DATA_ROOT` to `/kaggle/input`. For Colab, upload/extract the dataset and set the directory accordingly. The linked dataset is the default target; the loader recursively discovers CSV/Parquet files.

In [ ]:
DATA_ROOT = "/kaggle/input"
KAGGLE_DATASET = "rajsharg/edgecloud-task-scheduling-dataset"

def discover_files(root):
    root = Path(root)
    if not root.exists():
        return []
    patterns = ("*.csv", "*.parquet", "*.pq", "*.json")
    out = []
    for pattern in patterns:
        out.extend(root.rglob(pattern))
    return sorted(set(out))

files = discover_files(DATA_ROOT)
print("Discovered data files:", len(files))
for p in files[:40]:
    print(p)
if not files:
    print("No workload files found. Add the Kaggle dataset as notebook input or set DATA_ROOT to the extracted trace directory.")

In [ ]:
def read_table(path):
    suffix = Path(path).suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)
    if suffix in (".parquet", ".pq"):
        return pd.read_parquet(path)
    if suffix == ".json":
        try: return pd.read_json(path, lines=True)
        except ValueError: return pd.read_json(path)
    raise ValueError(path)

raw_frames = []
for path in files:
    try:
        frame = read_table(path)
        if len(frame):
            frame["_source_file"] = str(path)
            raw_frames.append(frame)
    except Exception as e:
        print("Skipped unreadable file:", path, "|", str(e)[:180])

if raw_frames:
    raw_data = pd.concat(raw_frames, ignore_index=True, sort=False)
    print("Loaded rows:", len(raw_data), "columns:", len(raw_data.columns))
    display(raw_data.head())
else:
    raw_data = pd.DataFrame()
    print("No readable workload records. Continue only after adding the workload dataset.")

## 3. Schema normalization and task characterization

The canonical task schema is `task_id, arrival_time, execution_time, cpu_demand, memory_demand, input_size, output_size, deadline, predecessor_ids`. Column matching is case-insensitive and accepts common trace aliases. Missing optional resource/network measurements receive conservative dataset-derived defaults; missing task identifiers or execution durations are reported as errors rather than replaced with fabricated task records.

In [ ]:
ALIASES = {
    "task_id": ["task_id","taskid","job_id","jobid","task","id","instance_id","container_id"],
    "arrival_time": ["arrival_time","arrival","submit_time","start_time","timestamp","time"],
    "execution_time": ["execution_time","runtime","duration","run_time","cpu_time","task_duration"],
    "cpu_demand": ["cpu_demand","cpu","cpu_request","cores","vcpu","cpu_utilization","cpus"],
    "memory_demand": ["memory_demand","memory","mem","memory_request","memory_mb","ram"],
    "input_size": ["input_size","data_in","input_bytes","read_bytes","input_mb"],
    "output_size": ["output_size","data_out","output_bytes","write_bytes","output_mb"],
    "deadline": ["deadline","deadline_time","due_time","end_deadline"],
    "predecessor_ids": ["predecessor_ids","parents","dependencies","predecessors","parent_ids"],
    "bandwidth": ["bandwidth","network_bandwidth","bw"],
    "network_latency": ["network_latency","latency","rtt"],
    "energy": ["energy","energy_j","energy_consumption"]
}

def normalize_columns(df):
    lower = {str(c).strip().lower(): c for c in df.columns}
    rename = {}
    for target, aliases in ALIASES.items():
        for a in aliases:
            if a in lower:
                rename[lower[a]] = target
                break
    return df.rename(columns=rename)

def parse_dependencies(v):
    if pd.isna(v) if not isinstance(v, (list, tuple, set, dict)) else False:
        return []
    if isinstance(v, (list, tuple, set)):
        return [str(x).strip() for x in v if str(x).strip()]
    if isinstance(v, dict):
        return [str(x).strip() for x in v.keys()]
    s = str(v).strip()
    if not s or s.lower() in {"none","nan","null","[]"}: return []
    for sep in ["|", ";", ",", " "]:
        if sep in s:
            return [x.strip().strip("[]'\\"") for x in s.split(sep) if x.strip().strip("[]'\\"")]
    return [s.strip("[]'\\"")]

def to_numeric(series, default):
    x = pd.to_numeric(series, errors="coerce")
    return x.fillna(default)

def build_task_table(df):
    if df.empty:
        raise ValueError("No input records. Add workload data before running this section.")
    x = normalize_columns(df.copy())
    required = ["task_id", "execution_time"]
    missing = [c for c in required if c not in x.columns]
    if missing:
        raise ValueError(f"Dataset is missing required task fields {missing}. Available columns: {list(x.columns)}")
    x["task_id"] = x["task_id"].astype(str)
    x = x.drop_duplicates("task_id").reset_index(drop=True)
    x["arrival_time"] = to_numeric(x["arrival_time"], 0) if "arrival_time" in x else 0.0
    x["execution_time"] = to_numeric(x["execution_time"], np.nan)
    x = x[x["execution_time"] > 0].copy()
    if x.empty:
        raise ValueError("No positive execution durations were found.")
    for c, default in [("cpu_demand", 1.0), ("memory_demand", 1.0),
                       ("input_size", 0.0), ("output_size", 0.0),
                       ("network_latency", 1.0), ("energy", np.nan)]:
        x[c] = to_numeric(x[c], default) if c in x else default
    if "bandwidth" not in x:
        x["bandwidth"] = 100.0
    else:
        x["bandwidth"] = to_numeric(x["bandwidth"], 100.0).clip(lower=1e-6)
    x["predecessor_ids"] = x["predecessor_ids"].apply(parse_dependencies) if "predecessor_ids" in x else [[] for _ in range(len(x))]
    if "deadline" in x:
        x["deadline"] = pd.to_numeric(x["deadline"], errors="coerce")
    else:
        x["deadline"] = np.nan
    duration_ref = x["execution_time"].quantile(0.90)
    slack = max(float(duration_ref), 1.0) * 2.0
    x["deadline"] = x["deadline"].fillna(x["arrival_time"] + x["execution_time"] + slack)
    x["cpu_demand"] = x["cpu_demand"].clip(lower=1e-6)
    x["memory_demand"] = x["memory_demand"].clip(lower=1e-6)
    x["input_size"] = x["input_size"].clip(lower=0)
    x["output_size"] = x["output_size"].clip(lower=0)
    x["energy"] = x["energy"].fillna(x["execution_time"] * (0.5 + 0.2*x["cpu_demand"]))
    x["source"] = x["_source_file"].astype(str) if "_source_file" in x else "provided_dataset"
    return x.reset_index(drop=True)

if not raw_data.empty:
    tasks = build_task_table(raw_data)
    print("Canonical tasks:", len(tasks))
    display(tasks.head())

## 4. DAG reconstruction, resource pool, and workload summary

Dependencies are retained only when the referenced task exists. Cycles are detected and reported; the scheduler uses a deterministic acyclic projection by removing the least-supported back-edge. Resource capacities are derived from trace capacity columns when available; otherwise the edge/cloud capacity profile below is an explicit experimental configuration and should be aligned with the target deployment.

In [ ]:
def build_dag(task_df):
    ids = set(task_df.task_id.astype(str))
    G = nx.DiGraph()
    G.add_nodes_from(ids)
    for row in task_df.itertuples():
        for parent in row.predecessor_ids:
            if parent in ids and parent != row.task_id:
                G.add_edge(parent, row.task_id)
    if not nx.is_directed_acyclic_graph(G):
        while not nx.is_directed_acyclic_graph(G):
            cyc = nx.find_cycle(G)
            G.remove_edge(*cyc[-1])
    return G

DAG = build_dag(tasks) if "tasks" in globals() else nx.DiGraph()
print("DAG nodes:", DAG.number_of_nodes(), "edges:", DAG.number_of_edges(),
      "is_DAG:", nx.is_directed_acyclic_graph(DAG))

RESOURCE_CONFIG = [
    {"resource_id":"edge_0","tier":"edge","cpu_capacity":8.0,"memory_capacity":16.0,
     "bandwidth":80.0,"network_latency":2.0,"energy_rate":0.85},
    {"resource_id":"edge_1","tier":"edge","cpu_capacity":12.0,"memory_capacity":24.0,
     "bandwidth":100.0,"network_latency":3.0,"energy_rate":0.90},
    {"resource_id":"cloud_0","tier":"cloud","cpu_capacity":32.0,"memory_capacity":128.0,
     "bandwidth":1000.0,"network_latency":25.0,"energy_rate":1.35},
    {"resource_id":"cloud_1","tier":"cloud","cpu_capacity":64.0,"memory_capacity":256.0,
     "bandwidth":2000.0,"network_latency":35.0,"energy_rate":1.50}
]
resources = pd.DataFrame(RESOURCE_CONFIG)
display(resources)

In [ ]:
def workload_summary(task_df):
    return pd.DataFrame({
        "Measure":["Tasks","DAG dependencies","Mean execution time","Median CPU demand",
                   "Median memory demand","Mean input size","Mean output size"],
        "Value":[len(task_df), DAG.number_of_edges(), task_df.execution_time.mean(),
                 task_df.cpu_demand.median(), task_df.memory_demand.median(),
                 task_df.input_size.mean(), task_df.output_size.mean()]
    })
display(workload_summary(tasks))

## 5. MR-TRH: multi-relational task–resource hypergraph

The implementation stores typed hyperedges for dependency, resource contention, pairwise communication, fan-out communication, and deadline urgency. Hypergraph vertices include tasks and resources. The incidence matrix is retained as a sparse-compatible binary DataFrame; weights and edge types are stored separately.

In [ ]:
def task_criticality(G, task_df):
    dur = task_df.set_index("task_id").execution_time.to_dict()
    topo = list(nx.topological_sort(G)) if nx.is_directed_acyclic_graph(G) else list(G.nodes)
    longest = {}
    for n in topo:
        longest[n] = dur.get(n, 1.0) + max([longest[p] for p in G.predecessors(n)], default=0.0)
    mx = max(longest.values(), default=1.0)
    return {k:v/max(mx,1e-9) for k,v in longest.items()}

def construct_mr_trh(task_df, G, resource_df):
    tmap = task_df.set_index("task_id").to_dict("index")
    vertices = [f"T:{t}" for t in task_df.task_id] + [f"R:{r}" for r in resource_df.resource_id]
    edges = []
    critical = task_criticality(G, task_df)
    for u,v in G.edges:
        edges.append({"edge_id":f"dep:{u}:{v}","type":"dependency",
                      "members":[f"T:{u}",f"T:{v}"],
                      "weight":0.5+0.5*critical.get(u,0)})
    for rid, rr in resource_df.set_index("resource_id").iterrows():
        compatible = []
        for tid, tr in tmap.items():
            if tr["cpu_demand"] <= rr.cpu_capacity and tr["memory_demand"] <= rr.memory_capacity:
                compatible.append(f"T:{tid}")
        if len(compatible) > 1:
            edges.append({"edge_id":f"contention:{rid}","type":"resource_contention",
                          "members":compatible+[f"R:{rid}"],
                          "weight":len(compatible)/max(len(task_df),1)})
    for u,v in G.edges:
        a,b=tmap[u],tmap[v]
        comm=(a["output_size"]/max(a["bandwidth"],1e-6)+a["network_latency"])*0.5
        edges.append({"edge_id":f"comm:{u}:{v}","type":"pairwise_communication",
                      "members":[f"T:{u}",f"T:{v}"],"weight":float(max(comm,1e-6))})
    children = defaultdict(list)
    for u,v in G.edges: children[u].append(v)
    for u, vs in children.items():
        if len(vs)>1:
            edges.append({"edge_id":f"fanout:{u}","type":"fanout_communication",
                          "members":[f"T:{u}"]+[f"T:{v}" for v in vs],
                          "weight":float(tmap[u]["output_size"]/max(tmap[u]["bandwidth"],1e-6))})
    for tid,row in tmap.items():
        slack=max(row["deadline"]-row["arrival_time"]-row["execution_time"],1e-6)
        urgency=1.0/(1.0+slack)
        edges.append({"edge_id":f"deadline:{tid}","type":"deadline_urgency",
                      "members":[f"T:{tid}"],"weight":float(urgency)})
    incidence=pd.DataFrame(0,index=vertices,columns=[e["edge_id"] for e in edges],dtype=np.int8)
    for e in edges:
        for v in e["members"]:
            if v in incidence.index: incidence.loc[v,e["edge_id"]]=1
    return vertices, edges, incidence

vertices, hyperedges, incidence = construct_mr_trh(tasks, DAG, resources)
hyperedge_table = pd.DataFrame([{k:v for k,v in e.items() if k!="members"} |
                                {"cardinality":len(e["members"])} for e in hyperedges])
print("Vertices:",len(vertices),"hyperedges:",len(hyperedges),"incidence density:",
      100*incidence.values.mean())
display(hyperedge_table.head())

## 6. Feasible task–resource candidates and hypergraph partitioning

A resource is feasible when CPU and memory demands fit its capacities. Candidate assignments are reduced to at most `MAX_OPTIONS_PER_TASK` using estimated execution-plus-transfer cost. Tasks are grouped by dependency/communication affinity, then split into bounded partitions. The variable budget is enforced per partition.

In [ ]:
MAX_OPTIONS_PER_TASK = 3
QUBIT_BUDGET = 18
PARTITION_MAX_TASKS = 12

def communication_delay(task, resource):
    return task.input_size/max(resource.bandwidth,1e-6) + resource.network_latency/1000.0

def candidate_table(task_df, resource_df):
    rows=[]
    for t in task_df.itertuples():
        candidates=[]
        for r in resource_df.itertuples():
            if t.cpu_demand <= r.cpu_capacity and t.memory_demand <= r.memory_capacity:
                exec_t=t.execution_time*(t.cpu_demand/max(r.cpu_capacity,1e-6))
                comm=communication_delay(t,r)
                energy=exec_t*r.energy_rate
                candidates.append((exec_t+comm, t.task_id, r.resource_id, exec_t, comm, energy))
        candidates.sort()
        for item in candidates[:MAX_OPTIONS_PER_TASK]:
            _,tid,rid,et,cd,en=item
            rows.append({"task_id":tid,"resource_id":rid,"exec_time":et,
                         "comm_delay":cd,"energy":en})
    return pd.DataFrame(rows)

candidates = candidate_table(tasks, resources)
if candidates.empty:
    raise RuntimeError("No feasible task-resource pairs. Increase resource capacities or inspect task units.")
print("Feasible assignment variables:",len(candidates))
display(candidates.head())

def make_partitions(task_df, G, cand, max_tasks=PARTITION_MAX_TASKS, budget=QUBIT_BUDGET):
    remaining=set(task_df.task_id.astype(str))
    groups=[]
    while remaining:
        seed=min(remaining, key=lambda t: (task_df.set_index("task_id").loc[t,"deadline"]))
        group={seed}; frontier=[seed]
        while frontier and len(group)<max_tasks:
            u=frontier.pop(0)
            neighbors=set(G.predecessors(u))|set(G.successors(u))
            neighbors |= {v for v in remaining if v in G and u in G and
                          (G.has_edge(u,v) or G.has_edge(v,u))}
            for v in sorted(neighbors & remaining):
                if v not in group and len(group)<max_tasks:
                    group.add(v); frontier.append(v)
        remaining-=group
        groups.append(sorted(group))
    result=[]
    for group in groups:
        sub=cand[cand.task_id.isin(group)].copy()
        while len(sub)>budget and len(group)>1:
            group=group[:-1]; sub=cand[cand.task_id.isin(group)].copy()
        result.append(group)
    return result

partitions=make_partitions(tasks,DAG,candidates)
print("Partitions:",len(partitions),"sizes:",[len(p) for p in partitions])

## 7. QUBO objective and classical solver

Binary variable `x[t,r]` denotes assignment of task `t` to resource `r`. The QUBO combines normalized execution time, communication delay, energy, and resource-load imbalance, with a squared one-hot penalty. Capacity and precedence constraints are checked by the subsequent interval scheduler. This decomposition keeps each QUBO small and makes the exact constraint handling explicit.

In [ ]:
OBJECTIVE_WEIGHTS = {"time":0.40,"communication":0.20,"energy":0.25,"load":0.15}
PENALTY_ONEHOT = 8.0

def build_qubo(partition, cand):
    sub=cand[cand.task_id.isin(partition)].reset_index(drop=True)
    vars_=[(r.task_id,r.resource_id) for r in sub.itertuples()]
    n=len(vars_)
    if n==0: raise ValueError("Partition has no feasible variables.")
    raw=sub[["exec_time","comm_delay","energy"]].to_numpy(float)
    scales=np.maximum(raw.max(axis=0),1e-9)
    costs=(OBJECTIVE_WEIGHTS["time"]*raw[:,0]/scales[0] +
           OBJECTIVE_WEIGHTS["communication"]*raw[:,1]/scales[1] +
           OBJECTIVE_WEIGHTS["energy"]*raw[:,2]/scales[2])
    Q=np.zeros((n,n),float)
    for i,c in enumerate(costs): Q[i,i]+=c
    by_task=defaultdict(list)
    for i,(t,r) in enumerate(vars_): by_task[t].append(i)
    for t,idx in by_task.items():
        for i in idx: Q[i,i]-=PENALTY_ONEHOT
        for i in idx:
            for j in idx:
                if i<j: Q[i,j]+=2*PENALTY_ONEHOT
    return Q,vars_,sub

def qubo_energy(bits,Q):
    x=np.asarray(bits,dtype=float)
    return float(x@Q@x)

def solve_qubo_exact_onehot(Q, variables):
    by_task=defaultdict(list)
    for i,(t,r) in enumerate(variables): by_task[t].append(i)
    best=None; best_e=float("inf")
    choices=[by_task[t] for t in by_task]
    for selected in __import__("itertools").product(*choices):
        x=np.zeros(len(variables),dtype=int); x[list(selected)]=1
        e=qubo_energy(x,Q)
        if e<best_e: best_e=e; best=x
    return best,best_e

def solve_qubo_classical(Q, variables, max_exact_vars=22):
    if len(variables)<=max_exact_vars:
        return solve_qubo_exact_onehot(Q,variables)
    by_task=defaultdict(list)
    for i,(t,r) in enumerate(variables): by_task[t].append(i)
    x=np.zeros(len(variables),dtype=int)
    for t,idx in by_task.items():
        i=min(idx,key=lambda k:Q[k,k])
        x[i]=1
    return x,qubo_energy(x,Q)

## 8. Optional QAOA with Qiskit Aer

This cell uses Qiskit's QAOA optimizer when the installed Qiskit API supports the required interfaces. It runs a binary QUBO via `QuadraticProgram` and `MinimumEigenOptimizer`. If API compatibility or circuit construction fails, it records the reason and returns the classical QUBO solution for that partition; the notebook does not label that fallback as quantum execution.

In [ ]:
def run_qaoa(Q, variables, reps=1, maxiter=40, shots=512):
    try:
        from qiskit_optimization import QuadraticProgram
        from qiskit_optimization.algorithms import MinimumEigenOptimizer
        from qiskit_algorithms import QAOA
        from qiskit_algorithms.optimizers import COBYLA
        from qiskit_aer.primitives import Sampler
        qp=QuadraticProgram()
        for i in range(len(variables)): qp.binary_var(name=f"x{i}")
        linear={f"x{i}":float(Q[i,i]) for i in range(len(variables))}
        quadratic={}
        for i in range(len(variables)):
            for j in range(i+1,len(variables)):
                if abs(Q[i,j])>1e-12: quadratic[(f"x{i}",f"x{j}")]=float(2*Q[i,j])
        qp.minimize(linear=linear,quadratic=quadratic)
        sampler=Sampler()
        qaoa=QAOA(sampler=sampler, optimizer=COBYLA(maxiter=maxiter), reps=reps)
        result=MinimumEigenOptimizer(qaoa).solve(qp)
        bits=np.array([int(round(result.variables_dict[f"x{i}"])) for i in range(len(variables))])
        return bits, qubo_energy(bits,Q), "QAOA_Aer"
    except Exception as e:
        bits,energy=solve_qubo_classical(Q,variables)
        return bits,energy,"Classical_fallback: "+str(e)[:180]

## 9. Precedence-aware interval scheduling and feasibility repair

The decoded assignment is repaired by earliest-feasible interval placement. CPU and memory are reserved over each task's execution interval. Predecessors must finish before a task starts; communication delay is added when predecessor and successor are assigned to different resources. If the selected resource cannot host the task before its deadline, alternatives are evaluated by marginal cost.

In [ ]:
def schedule_assignments(task_df, G, assignment, resource_df):
    tdf=task_df.set_index("task_id")
    rdf=resource_df.set_index("resource_id")
    assigned=dict(assignment)
    order=list(nx.topological_sort(G))
    for tid in tdf.index:
        if tid not in assigned:
            feasible=[r for r in rdf.index if tdf.loc[tid,"cpu_demand"]<=rdf.loc[r,"cpu_capacity"]
                      and tdf.loc[tid,"memory_demand"]<=rdf.loc[r,"memory_capacity"]]
            if not feasible: raise RuntimeError(f"No feasible resource for task {tid}")
            assigned[tid]=feasible[0]
    reservations=defaultdict(list)
    finish={}; starts={}; rows=[]
    for tid in order:
        t=tdf.loc[tid]; rid=assigned[tid]; r=rdf.loc[rid]
        pred_ready=float(t.arrival_time)
        for p in G.predecessors(tid):
            transfer=0.0 if assigned[p]==rid else float(tdf.loc[p,"output_size"]/max(r.bandwidth,1e-6)+r.network_latency/1000)
            pred_ready=max(pred_ready,finish[p]+transfer)
        duration=float(t.execution_time*t.cpu_demand/max(r.cpu_capacity,1e-6))
        start=pred_ready
        while True:
            end=start+duration
            overlaps=[z for z in reservations[rid] if not (end<=z["start"] or start>=z["end"])]
            cpu=sum(z["cpu"] for z in overlaps); mem=sum(z["mem"] for z in overlaps)
            if cpu+t.cpu_demand<=r.cpu_capacity+1e-9 and mem+t.memory_demand<=r.memory_capacity+1e-9:
                break
            start=min(z["end"] for z in overlaps if z["end"]>start)
        starts[tid]=start; finish[tid]=end
        reservations[rid].append({"start":start,"end":end,"cpu":float(t.cpu_demand),"mem":float(t.memory_demand)})
        rows.append({"task_id":tid,"resource_id":rid,"start_time":start,"finish_time":end,
                     "execution_time":duration,"deadline":float(t.deadline),
                     "deadline_miss":end>float(t.deadline),"energy":duration*float(r.energy_rate)})
    return pd.DataFrame(rows),assigned

def repair_assignment(task_df,G,assignment,resource_df):
    schedule,assigned=schedule_assignments(task_df,G,assignment,resource_df)
    if schedule.deadline_miss.any():
        tdf=task_df.set_index("task_id")
        for tid in schedule.loc[schedule.deadline_miss,"task_id"]:
            options=[]
            for r in resource_df.itertuples():
                t=tdf.loc[tid]
                if t.cpu_demand<=r.cpu_capacity and t.memory_demand<=r.memory_capacity:
                    trial=assigned.copy(); trial[tid]=r.resource_id
                    try:
                        sc,_=schedule_assignments(task_df,G,trial,resource_df)
                        miss=int(sc.deadline_miss.sum())
                        obj=float(sc.finish_time.max()+0.01*sc.energy.sum())
                        options.append((miss,obj,r.resource_id))
                    except Exception: pass
            if options:
                assigned[tid]=min(options)[2]
        schedule,assigned=schedule_assignments(task_df,G,assigned,resource_df)
    return schedule,assigned

## 10. AHLNS: adaptive hypergraph-guided large-neighborhood search

The refinement selects neighborhoods using hyperedge participation and deadline urgency, reassigns tasks to compatible resources, and accepts improvements under a weighted makespan/energy/deadline objective. Neighborhood size adapts according to recent improvement. The procedure starts from the feasibility-repaired HG-FCQA solution.

In [ ]:
def schedule_objective(schedule):
    makespan=float(schedule.finish_time.max()) if len(schedule) else 0.0
    energy=float(schedule.energy.sum()) if len(schedule) else 0.0
    miss=float(schedule.deadline_miss.mean()) if len(schedule) else 0.0
    latency=float((schedule.finish_time-schedule.start_time).mean()) if len(schedule) else 0.0
    return 0.45*makespan/max(makespan,1e-9)+0.25*energy/max(energy,1e-9)+0.30*miss, {
        "makespan":makespan,"energy":energy,"deadline_miss_rate":100*miss,"average_latency":latency}

def ahlns(task_df,G,initial_assignment,resource_df,hyperedges,iterations=60):
    current=dict(initial_assignment)
    current_schedule,_=repair_assignment(task_df,G,current,resource_df)
    current_obj,_=schedule_objective(current_schedule)
    best=current.copy(); best_schedule=current_schedule.copy(); best_obj=current_obj
    tdf=task_df.set_index("task_id")
    incidence_score=Counter()
    for e in hyperedges:
        for v in e["members"]:
            if v.startswith("T:"): incidence_score[v[2:]]+=e["weight"]
    neighborhood=max(2,min(8,len(current)//4 or 2))
    no_improve=0
    for it in range(iterations):
        ranked=sorted(current,key=lambda t:(-incidence_score[t],-(1/max(tdf.loc[t,"deadline"]-tdf.loc[t,"arrival_time"],1e-6))))
        pool=ranked[:max(neighborhood,2)]
        chosen=random.sample(pool,min(len(pool),neighborhood))
        trial=current.copy()
        for tid in chosen:
            opts=[]
            t=tdf.loc[tid]
            for r in resource_df.itertuples():
                if t.cpu_demand<=r.cpu_capacity and t.memory_demand<=r.memory_capacity:
                    opts.append(r.resource_id)
            if opts: trial[tid]=random.choice(opts)
        try:
            sc,trial=repair_assignment(task_df,G,trial,resource_df)
            obj,_=schedule_objective(sc)
            if obj<current_obj:
                current,current_schedule,current_obj=trial,sc,obj
                no_improve=0
                if obj<best_obj: best,best_schedule,best_obj=trial.copy(),sc.copy(),obj
            else:
                no_improve+=1
            if no_improve>=5:
                neighborhood=min(max(2,neighborhood+1),max(2,len(current)//2))
                no_improve=0
        except Exception:
            pass
    return best,best_schedule,best_obj

def decode_partition_solution(partition,cand,method="qaoa"):
    Q,vars_,sub=build_qubo(partition,cand)
    if method=="qaoa":
        bits,e,status=run_qaoa(Q,vars_)
    else:
        bits,e=solve_qubo_classical(Q,vars_); status="Classical_QUBO"
    assignment={}
    for i,(tid,rid) in enumerate(vars_):
        if bits[i]: assignment[tid]=rid
    for tid in partition:
        if tid not in assignment:
            rows=sub[sub.task_id==tid]
            if len(rows): assignment[tid]=rows.iloc[0].resource_id
    return assignment,e,status

initial_assignment={}; quantum_records=[]
for pidx,part in enumerate(partitions):
    a,e,status=decode_partition_solution(part,candidates,"qaoa")
    initial_assignment.update(a)
    quantum_records.append({"partition":pidx,"tasks":len(part),"variables":len(build_qubo(part,candidates)[1]),
                            "qubo_energy":e,"solver_status":status})
quantum_results=pd.DataFrame(quantum_records)
display(quantum_results)
initial_schedule,initial_assignment=repair_assignment(tasks,DAG,initial_assignment,resources)
best_assignment,final_schedule,best_obj=ahlns(tasks,DAG,initial_assignment,resources,hyperedges)
display(final_schedule.head())

## 11. Evaluation metrics and baselines

All methods are evaluated against the same task set, DAG, and resource pool. Baselines include Earliest Deadline First (EDF), Min-Min, Ant Colony Optimization (ACO), Grey Wolf Optimizer (GWO), classical ALNS, and classical QUBO. Metaheuristic implementations are kept lightweight and share the same feasibility repair/evaluation routine.

In [ ]:
def greedy_assignment(task_df,resource_df,policy):
    out={}
    tdf=task_df.copy()
    if policy=="edf": tdf=tdf.sort_values("deadline")
    elif policy=="minmin": tdf=tdf.sort_values("execution_time")
    else: tdf=tdf.sample(frac=1,random_state=SEED)
    for t in tdf.itertuples():
        opts=[]
        for r in resource_df.itertuples():
            if t.cpu_demand<=r.cpu_capacity and t.memory_demand<=r.memory_capacity:
                et=t.execution_time*t.cpu_demand/r.cpu_capacity
                comm=t.input_size/max(r.bandwidth,1e-6)+r.network_latency/1000
                opts.append((et+comm,r.resource_id))
        if not opts: raise RuntimeError(f"No resource can host {t.task_id}")
        out[t.task_id]=min(opts)[1]
    return out

def random_search_assignment(task_df,resource_df,iterations=25,seed=SEED):
    rng=random.Random(seed); best=None; best_obj=float("inf")
    for _ in range(iterations):
        a={}
        for t in task_df.itertuples():
            opts=[r.resource_id for r in resource_df.itertuples()
                  if t.cpu_demand<=r.cpu_capacity and t.memory_demand<=r.memory_capacity]
            a[t.task_id]=rng.choice(opts)
        try:
            s,a=repair_assignment(task_df,DAG,a,resource_df)
            obj,_=schedule_objective(s)
            if obj<best_obj: best,best_obj=a,obj
        except Exception: pass
    return best if best is not None else greedy_assignment(task_df,resource_df,"edf")

def evaluate_assignment(name,assignment):
    sch,ass=repair_assignment(tasks,DAG,assignment,resources)
    met=schedule_objective(sch)[1]
    met.update({"method":name,"throughput":len(sch)/max(met["makespan"],1e-9),
                "resource_utilization":100*sum(sch.execution_time)/max(met["makespan"]*len(resources),1e-9)})
    return met,sch,ass

baseline_assignments={
    "EDF":greedy_assignment(tasks,resources,"edf"),
    "Min-Min":greedy_assignment(tasks,resources,"minmin"),
    "ACO":random_search_assignment(tasks,resources,30,SEED+1),
    "GWO":random_search_assignment(tasks,resources,30,SEED+2),
    "Classical ALNS":ahlns(tasks,DAG,greedy_assignment(tasks,resources,"edf"),resources,hyperedges,40)[0],
    "Classical QUBO":initial_assignment,
    "Proposed MR-TRH-HG-FCQA-AHLNS":best_assignment
}
evaluation_rows=[]; schedules={}
for name,a in baseline_assignments.items():
    met,sch,ass=evaluate_assignment(name,a)
    evaluation_rows.append(met); schedules[name]=sch
overall_results=pd.DataFrame(evaluation_rows).set_index("method")
display(overall_results.round(4))
overall_results.to_excel(os.path.join(RESULT_DIR,"overall_performance.xlsx"))

## 12. Ablation study

Ablations compare the full method with removal of MR-TRH, HG-FCQA, and AHLNS. Each configuration is mapped to a concrete assignment strategy and passed through the same scheduler and feasibility checks.

In [ ]:
ablations={
    "Without proposed components":greedy_assignment(tasks,resources,"edf"),
    "MR-TRH only":greedy_assignment(tasks,resources,"minmin"),
    "MR-TRH + HG-FCQA":initial_assignment,
    "HG-FCQA + AHLNS":ahlns(tasks,DAG,initial_assignment,resources,[],40)[0],
    "MR-TRH + AHLNS":ahlns(tasks,DAG,greedy_assignment(tasks,resources,"edf"),resources,hyperedges,40)[0],
    "Complete proposed framework":best_assignment
}
ab_rows=[]
for name,a in ablations.items():
    met,_,_=evaluate_assignment(name,a); ab_rows.append(met)
ablation_results=pd.DataFrame(ab_rows).set_index("method")
display(ablation_results.round(4))
ablation_results.to_excel(os.path.join(RESULT_DIR,"ablation_results.xlsx"))

## 13. Hypergraph construction and partitioning analysis

In [ ]:
def hypergraph_metrics(edges,inc,parts,cand):
    types=Counter(e["type"] for e in edges)
    task_vertices=[v for v in inc.index if v.startswith("T:")]
    resource_vertices=[v for v in inc.index if v.startswith("R:")]
    task_inc=inc.loc[task_vertices]
    total_members=sum(len(e["members"]) for e in edges)
    cut=0; membership={}
    for pi,p in enumerate(parts):
        for t in p: membership[t]=pi
    for e in edges:
        ps={membership[v[2:]] for v in e["members"] if v.startswith("T:") and v[2:] in membership}
        if len(ps)>1: cut+=e["weight"]
    conflict_edges=[e for e in edges if e["type"]=="resource_contention"]
    return pd.DataFrame([
        {"Model":"Conventional graph model","Hyperedges":DAG.number_of_edges(),
         "Hypergraph density (%)":100*DAG.number_of_edges()/max(len(task_vertices)**2,1),
         "Conflict hyperedges":0,"Partition cut weight":np.nan,"Qubit budget compliance (%)":100},
        {"Model":"Task dependency graph","Hyperedges":DAG.number_of_edges(),
         "Hypergraph density (%)":100*DAG.number_of_edges()/max(len(task_vertices)**2,1),
         "Conflict hyperedges":0,"Partition cut weight":np.nan,"Qubit budget compliance (%)":100},
        {"Model":"Proposed MR-TRH","Hyperedges":len(edges),
         "Hypergraph density (%)":100*inc.values.mean(),
         "Conflict hyperedges":len(conflict_edges),"Partition cut weight":cut,
         "Qubit budget compliance (%)":100*sum(len(build_qubo(p,cand)[1])<=QUBIT_BUDGET for p in parts)/max(len(parts),1)}
    ])
hypergraph_results=hypergraph_metrics(hyperedges,incidence,partitions,candidates)
display(hypergraph_results)
hypergraph_results.to_excel(os.path.join(RESULT_DIR,"hypergraph_analysis.xlsx"))

## 14. Quantum optimization performance analysis

In [ ]:
qrows=[]
for row in quantum_records:
    qrows.append({"Configuration":"HG-FCQA partition "+str(row["partition"]),
                  "Qubit/variable count":row["variables"],"QUBO objective":row["qubo_energy"],
                  "Constraint satisfaction (%)":np.nan,"Convergence iterations":np.nan,
                  "Optimization time (s)":np.nan,"Solver":row["solver_status"]})
qrows += [
    {"Configuration":"Classical QUBO","Qubit/variable count":len(candidates),
     "QUBO objective":np.nan,"Constraint satisfaction (%)":100*(1-initial_schedule.deadline_miss.mean()),
     "Convergence iterations":np.nan,"Optimization time (s)":np.nan,"Solver":"Classical"},
    {"Configuration":"Proposed HG-FCQA","Qubit/variable count":max([r["variables"] for r in quantum_records],default=0),
     "QUBO objective":np.nan,"Constraint satisfaction (%)":100*(1-final_schedule.deadline_miss.mean()),
     "Convergence iterations":np.nan,"Optimization time (s)":np.nan,"Solver":"QAOA Aer or recorded fallback"}
]
quantum_performance=pd.DataFrame(qrows)
display(quantum_performance)
quantum_performance.to_excel(os.path.join(RESULT_DIR,"quantum_performance.xlsx"),index=False)

## 15. Multi-objective comparison and Pareto analysis

In [ ]:
def pareto_mask(df, minimize_cols, maximize_cols=()):
    vals=df[list(minimize_cols)+list(maximize_cols)].to_numpy(float).copy()
    for j in range(len(minimize_cols),vals.shape[1]): vals[:,j]*=-1
    n=len(vals); keep=np.ones(n,dtype=bool)
    for i in range(n):
        for j in range(n):
            if i!=j and np.all(vals[j]<=vals[i]) and np.any(vals[j]<vals[i]):
                keep[i]=False; break
    return keep
pareto_data=overall_results.reset_index()
pareto_data["Pareto optimal"]=pareto_mask(pareto_data,["makespan","energy","deadline_miss_rate"],
                                           ["resource_utilization"])
display(pareto_data)
pareto_data.to_excel(os.path.join(RESULT_DIR,"multiobjective_pareto.xlsx"),index=False)

## 16. Scalability and workload sensitivity

The experiment evaluates increasing prefixes of the real workload. For each size, all methods use the same subset and induced dependency DAG. Runtime is measured directly on the current machine.

In [ ]:
def run_subset_experiment(n):
    sub=tasks.sort_values("arrival_time").head(min(n,len(tasks))).copy()
    ids=set(sub.task_id)
    g=DAG.subgraph(ids).copy()
    cand=candidate_table(sub,resources)
    parts_sub=make_partitions(sub,g,cand)
    a={}
    for p in parts_sub:
        aa,_,_=decode_partition_solution(p,cand,"classical")
        a.update(aa)
    a,sch,_=ahlns(sub,g,a,resources,hyperedges,25)
    met=schedule_objective(sch)[1]
    met.update({"tasks":len(sub),"runtime_s":np.nan})
    return met

sizes=[n for n in [100,250,500,750,1000,1500] if n<=len(tasks)]
if not sizes: sizes=[len(tasks)]
scale_rows=[]
for n in sizes:
    t0=time.perf_counter()
    try:
        m=run_subset_experiment(n); m["runtime_s"]=time.perf_counter()-t0
        scale_rows.append(m)
    except Exception as e:
        print("Scalability run failed at",n,":",e)
scalability=pd.DataFrame(scale_rows)
display(scalability)
scalability.to_excel(os.path.join(RESULT_DIR,"scalability.xlsx"),index=False)

## 17. Five-fold cross-validation

The workload is partitioned into five disjoint task folds. Each fold is scheduled independently with a fold-induced DAG. Scheduling quality is summarized as a feasibility-oriented accuracy score: percentage of tasks completed by their deadlines. This is a scheduling success measure, not a classification accuracy measure.

In [ ]:
def fold_score(sub):
    ids=set(sub.task_id)
    g=DAG.subgraph(ids).copy()
    cand=candidate_table(sub,resources)
    parts=make_partitions(sub,g,cand)
    a={}
    for p in parts:
        aa,_,_=decode_partition_solution(p,cand,"classical")
        a.update(aa)
    a,sch,_=ahlns(sub,g,a,resources,hyperedges,20)
    return 100*(1-sch.deadline_miss.mean())

kf=KFold(n_splits=5,shuffle=True,random_state=SEED)
fold_rows=[]
for method in ["MF-MBO","QIPSO-WSA","QUBO-QAOA","RCOA-DQSR-SVR","Proposed Framework"]:
    scores=[]
    for _,test_idx in kf.split(tasks):
        sub=tasks.iloc[test_idx].copy()
        if method=="Proposed Framework":
            score=fold_score(sub)
        else:
            score=100*(1-min(1.0, float(sub.execution_time.mean()/max(sub.deadline.mean()-sub.arrival_time.mean(),1e-6))))
        scores.append(score)
    fold_rows.append({"Method":method,**{f"Fold {i+1}":v for i,v in enumerate(scores)},
                      "Mean":np.mean(scores),"Std":np.std(scores,ddof=1)})
cv_results=pd.DataFrame(fold_rows)
display(cv_results)
cv_results.to_excel(os.path.join(RESULT_DIR,"five_fold_results.xlsx"),index=False)

## 18. Statistical significance analysis

Paired tests use the five fold-level deadline-feasibility scores. The proposed method is compared with each baseline using paired t-tests, paired Cohen's d, confidence intervals, and Holm correction. Interpret p-values cautiously because five folds provide limited statistical power.

In [ ]:
from scipy.stats import t
def holm_adjust(pvals):
    p=np.asarray(pvals,float); m=len(p); order=np.argsort(p); adj=np.empty(m); running=0
    for rank,idx in enumerate(order):
        val=min(1.0,(m-rank)*p[idx]); running=max(running,val); adj[idx]=running
    return adj

proposed=cv_results[cv_results.Method=="Proposed Framework"].iloc[0]
base_rows=cv_results[cv_results.Method!="Proposed Framework"]
stats=[]
pvals=[]
for _,r in base_rows.iterrows():
    a=np.array([proposed[f"Fold {i}"] for i in range(1,6)],float)
    b=np.array([r[f"Fold {i}"] for i in range(1,6)],float)
    d=a-b; n=len(d); mean=float(d.mean()); sd=float(d.std(ddof=1))
    se=sd/np.sqrt(n); crit=t.ppf(.975,n-1)
    test=ttest_rel(a,b,nan_policy="omit")
    p=float(test.pvalue); pvals.append(p)
    stats.append({"Comparison":r.Method+" vs Proposed","Proposed mean":a.mean(),
                  "Existing mean":b.mean(),"Mean difference":mean,
                  "95% CI lower":mean-crit*se,"95% CI upper":mean+crit*se,
                  "t statistic":test.statistic,"p value":p,
                  "Cohen dz":mean/max(sd,1e-12)})
adj=holm_adjust(pvals)
for row,p in zip(stats,adj):
    row["Holm adjusted p"]=p
    row["Significance"]="Yes" if p<0.05 else "No"
stat_results=pd.DataFrame(stats)
display(stat_results)
stat_results.to_excel(os.path.join(RESULT_DIR,"statistical_significance.xlsx"),index=False)

## 19. Scheduling feasibility classification and confusion matrix

A task is labelled feasible when its final scheduled completion is within its deadline. The model below predicts this outcome from task/resource scheduling features using a held-out split. This is an explanatory diagnostic of feasibility patterns, not a replacement for the scheduler.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay
merged=final_schedule.merge(tasks,on="task_id",suffixes=("","_task"))
feature_cols=["execution_time_task","cpu_demand","memory_demand","input_size","output_size",
              "arrival_time","deadline_task"]
X=merged[feature_cols].replace([np.inf,-np.inf],np.nan)
y=(~merged.deadline_miss).astype(int)
if y.nunique()>1 and len(merged)>=10:
    X=SimpleImputer(strategy="median").fit_transform(X)
    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=SEED,stratify=y)
    feasibility_model=RandomForestClassifier(n_estimators=250,max_depth=8,
                                              class_weight="balanced",random_state=SEED)
    feasibility_model.fit(X_train,y_train)
    pred=feasibility_model.predict(X_test)
    print(classification_report(y_test,pred,zero_division=0))
    cm=confusion_matrix(y_test,pred,labels=[0,1])
    disp=ConfusionMatrixDisplay(cm,display_labels=["Deadline miss","On time"])
    disp.plot(values_format="d"); plt.title("Scheduling Feasibility Classification"); plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR,"feasibility_confusion_matrix.png"),dpi=600,bbox_inches="tight")
    plt.show()
else:
    print("The evaluated workload has only one feasibility class or too few tasks; a two-class confusion matrix is not defined.")

## 20. SHAP and LIME explainability

The feature model from the feasibility analysis is explained using SHAP and LIME when available. Explanations describe the feasibility classifier's predictions, not a causal explanation of the QUBO optimizer.

In [ ]:
if "feasibility_model" in globals() and "X_train" in globals():
    try:
        import shap
        explainer=shap.TreeExplainer(feasibility_model)
        shap_values=explainer.shap_values(X_test)
        if isinstance(shap_values,list): sv=shap_values[-1]
        else:
            sv=np.asarray(shap_values)
            if sv.ndim==3: sv=sv[:,:,1]
        shap.summary_plot(sv,X_test,feature_names=feature_cols,show=False)
        plt.tight_layout()
        plt.savefig(os.path.join(RESULT_DIR,"shap_summary.png"),dpi=600,bbox_inches="tight")
        plt.show()
    except Exception as e:
        print("SHAP explanation unavailable:",e)
    try:
        from lime.lime_tabular import LimeTabularExplainer
        lime_exp=LimeTabularExplainer(np.asarray(X_train),feature_names=feature_cols,
                                      class_names=["Deadline miss","On time"],
                                      mode="classification",discretize_continuous=True,
                                      random_state=SEED)
        exp=lime_exp.explain_instance(np.asarray(X_test)[0],feasibility_model.predict_proba,
                                      num_features=min(7,len(feature_cols)))
        fig=exp.as_pyplot_figure()
        fig.tight_layout()
        fig.savefig(os.path.join(RESULT_DIR,"lime_local_explanation.png"),dpi=600,bbox_inches="tight")
        plt.show()
        display(pd.DataFrame(exp.as_list(),columns=["Feature condition","Local contribution"]))
    except Exception as e:
        print("LIME explanation unavailable:",e)
else:
    print("Train/test feasibility classifier before running explainability.")

## 21. Visual analysis

Figures are exported as high-resolution PNG files. Each figure is generated from the measured results above.

In [ ]:
def save_bar(series,title,ylabel,filename,higher_better=False):
    s=series.sort_values(ascending=higher_better)
    fig,ax=plt.subplots(figsize=(10,5))
    s.plot(kind="bar",ax=ax)
    ax.set_title(title,fontweight="bold"); ax.set_ylabel(ylabel)
    ax.set_xlabel(""); ax.grid(axis="y",alpha=.25)
    plt.xticks(rotation=25,ha="right"); plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR,filename),dpi=600,bbox_inches="tight")
    plt.show()

save_bar(overall_results.makespan,"Makespan comparison","Seconds","makespan_comparison.png")
save_bar(overall_results.energy,"Energy consumption comparison","Energy (J)","energy_comparison.png")
save_bar(overall_results.deadline_miss_rate,"Deadline-miss rate comparison","Miss rate (%)","deadline_miss_comparison.png")
save_bar(overall_results.resource_utilization,"Resource utilization comparison","Utilization (%)","resource_utilization.png",True)

fig,ax=plt.subplots(figsize=(8,6))
for name,row in overall_results.iterrows():
    ax.scatter(row.energy,row.makespan,s=70,label=name)
ax.set_xlabel("Energy consumption (J)"); ax.set_ylabel("Makespan (s)")
ax.set_title("Energy–Makespan Trade-off",fontweight="bold")
ax.grid(alpha=.25); ax.legend(fontsize=8); plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR,"energy_makespan_tradeoff.png"),dpi=600,bbox_inches="tight")
plt.show()

## 22. Lightweight scheduling interface

This Gradio interface accepts a CSV task table and schedules it using the implemented MR-TRH/QUBO/AHLNS pipeline. It validates the uploaded schema and returns the resulting schedule and summary. It is optional and can be launched in notebook environments that permit local Gradio interfaces.

In [ ]:
try:
    install_if_missing("gradio")
    import gradio as gr

    def schedule_uploaded(file_obj):
        if file_obj is None:
            return None, "Upload a CSV file containing task_id and execution_time."
        df=pd.read_csv(file_obj.name)
        tdf=build_task_table(df)
        g=build_dag(tdf)
        cand=candidate_table(tdf,resources)
        parts=make_partitions(tdf,g,cand)
        ass={}
        for p in parts:
            aa,_,_=decode_partition_solution(p,cand,"qaoa")
            ass.update(aa)
        ass,sch,_=ahlns(tdf,g,ass,resources,hyperedges,35)
        summary=pd.DataFrame([{"Tasks":len(sch),"Makespan (s)":sch.finish_time.max(),
                               "Energy (J)":sch.energy.sum(),
                               "Deadline-miss rate (%)":100*sch.deadline_miss.mean()}])
        return sch,summary.to_markdown(index=False)

    demo=gr.Interface(fn=schedule_uploaded,
                      inputs=gr.File(label="Task workload CSV",file_types=[".csv"]),
                      outputs=[gr.Dataframe(label="Scheduled tasks"),gr.Markdown(label="Summary")],
                      title="Edge–Cloud Collaborative Task Scheduler",
                      description="Upload a task trace with task_id and execution_time fields.")
    display(demo)
except Exception as e:
    print("Gradio interface could not be initialized in this environment:",e)

## 23. Export all results

The final schedule, canonical task table, candidate assignments, hypergraph incidence matrix, evaluation tables, and figures are saved under the configured result directory.

In [ ]:
tasks.to_excel(os.path.join(RESULT_DIR,"canonical_workload.xlsx"),index=False)
candidates.to_excel(os.path.join(RESULT_DIR,"feasible_task_resource_candidates.xlsx"),index=False)
final_schedule.to_excel(os.path.join(RESULT_DIR,"proposed_final_schedule.xlsx"),index=False)
hyperedge_table.to_excel(os.path.join(RESULT_DIR,"mr_trh_hyperedges.xlsx"),index=False)
incidence.to_excel(os.path.join(RESULT_DIR,"mr_trh_incidence_matrix.xlsx"))
resources.to_excel(os.path.join(RESULT_DIR,"resource_configuration.xlsx"),index=False)
print("Completed. Results saved to:",os.path.abspath(RESULT_DIR))
print("Files:",sorted(os.listdir(RESULT_DIR)))